In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# 1) Filter: drop users with < 200 ratings, then books with < 100 ratings.
# Book counts are recomputed AFTER the user filter, so every remaining book
# has >= 100 ratings from the remaining (active) users.
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index
df_filtered = df_ratings[df_ratings['user'].isin(active_users)]

book_counts = df_filtered['isbn'].value_counts()
popular_isbns = book_counts[book_counts >= 100].index
df_filtered = df_filtered[df_filtered['isbn'].isin(popular_isbns)]

print(f"ratings: {len(df_ratings):,} -> {len(df_filtered):,}")
print(f"users:   {df_ratings['user'].nunique():,} -> {df_filtered['user'].nunique():,}")
print(f"books:   {df_ratings['isbn'].nunique():,} -> {df_filtered['isbn'].nunique():,}")

In [ ]:
# 2) Attach titles. Several ISBNs can share one title (different editions), and
# the same user may have rated more than one of them -> keep one rating per (user, title),
# otherwise pivot() raises "Index contains duplicate entries".
df = df_filtered.merge(df_books[['isbn', 'title']].drop_duplicates('isbn'), on='isbn', how='inner')
df = df.drop_duplicates(subset=['user', 'title'])
print(df.shape)

In [ ]:
# 3) Item-user matrix (rows = books, columns = users), missing = 0, stored sparse.
book_user = df.pivot(index='title', columns='user', values='rating').fillna(0)
titles = book_user.index
X = csr_matrix(book_user.values)

# 4) Cosine distance, brute force. KD-tree / ball-tree do not support the cosine metric,
# and brute force is cheap at this size (~hundreds of books x ~900 users).
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(X)
print(X.shape)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  if book not in titles:
    raise ValueError(f"'{book}' is not in the filtered dataset "
                     "(it may have < 100 ratings or no rater with >= 200 ratings).")
  idx = titles.get_loc(book)

  # ask for 6 neighbours: the query book itself is always one of them
  distances, indices = model.kneighbors(X[idx], n_neighbors=6)

  recs = [[titles[i], float(d)]
          for d, i in zip(distances[0], indices[0])
          if i != idx][:5]

  # kneighbors returns ascending distance; the expected output is farthest -> nearest
  recs = recs[::-1]

  recommended_books = [book, recs]
  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()